# Bag-of-Words SL (word-heteroskedastic)

In [ ]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
    mse_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:0")

## Configure

In [ ]:
config = BagOfWordsSLConfig.get_canonical(
    dataset="word_heteroskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-word-het-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
)

display(config.visualize())

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

In [ ]:
state.run_training()

## Results

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

In [ ]:
analysis = BagOfWordsAnalysisConfig.from_grouped({
    "example": [(0, config.study_folder)]
})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()